# Preparação dos landmarks faciais para modelagem

Este notebook reúne os três CSVs de `data/data_preps`, identifica o participante de origem e prepara os dados em cinco etapas:

1. inclusão de `participant_id` e harmonização dos rótulos;
2. união e auditoria de valores ausentes, infinitos e duplicatas;
3. limpeza das observações inválidas;
4. normalização geométrica dos 468 landmarks;
5. criação de atributos de distâncias entre regiões das sobrancelhas, olhos, bochechas, lábios e queixo.

As coordenadas do CSV já foram centralizadas na ponta do nariz durante a coleta. Aqui elas são novamente ancoradas no landmark 1 (por segurança) e divididas pela distância entre os cantos externos dos olhos (landmarks 33 e 263). Assim, as medidas ficam adimensionais e menos sensíveis à posição da pessoa e à distância até a câmera, sem estimar parâmetros com o conjunto completo e sem introduzir vazamento de dados.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.5f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PREPS_DIR = PROJECT_ROOT / "data" / "data_preps"
DATASETS = {
    DATA_PREPS_DIR / "luiz_collected_landmarks.csv": 1,
    DATA_PREPS_DIR / "jonatas_collected_landmark.csv": 2,
    DATA_PREPS_DIR / "caua_collected_landmarks.csv": 3,
}
PARTICIPANT_NAMES = {1: "luiz", 2: "jonatas", 3: "caua"}
OUTPUT_PATH = PROJECT_ROOT / "data" / "prepared_landmarks.csv"

missing_files = [path for path in DATASETS if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Arquivo(s) não encontrado(s): {missing_files}")

pd.DataFrame({
    "participant_id": list(DATASETS.values()),
    "participante": [PARTICIPANT_NAMES[value] for value in DATASETS.values()],
    "arquivo": [path.name for path in DATASETS],
})

## 1. Carregamento e inspeção inicial

Cada arquivo recebe o `participant_id` **antes da concatenação**: Luiz = 1, Jonatas = 2 e Cauã = 3. As coordenadas são carregadas como `float32` para reduzir o consumo de memória. Também harmonizamos os nomes equivalentes das três classes para `boca_aberta`, `neutro` e `sobrancelha`.

In [ ]:
source_metadata_columns = ["label", "session_id", "frame_index"]
metadata_columns = ["label", "participant_id", "session_id", "frame_index"]
first_header = pd.read_csv(next(iter(DATASETS)), nrows=0).columns.tolist()
coordinate_columns = [
    column for column in first_header if column not in source_metadata_columns
]

expected_coordinate_columns = [
    f"{axis}_{index}"
    for index in range(468)
    for axis in ("x", "y", "z")
]
expected_coordinate_count = len(expected_coordinate_columns)
if coordinate_columns != expected_coordinate_columns:
    raise ValueError(
        f"Esperadas {expected_coordinate_count} coordenadas na ordem x/y/z "
        f"dos 468 landmarks, mas o cabeçalho encontrado é incompatível."
    )

for path in DATASETS:
    if pd.read_csv(path, nrows=0).columns.tolist() != first_header:
        raise ValueError(f"O cabeçalho de {path.name} difere dos demais arquivos.")

LABEL_MAPPING = {
    "boca_aberta": "boca_aberta",
    "boca": "boca_aberta",
    "mouth_open": "boca_aberta",
    "neutro": "neutro",
    "neutral": "neutro",
    "sobrancelha": "sobrancelha",
    "surpresa": "sobrancelha",
    "raise_eyebrows": "sobrancelha",
}

dtypes = {column: "float32" for column in coordinate_columns}
dtypes.update({"label": "string", "session_id": "string", "frame_index": "Int64"})
frames = []
source_report = []

for path, participant_id in DATASETS.items():
    participant_df = pd.read_csv(path, dtype=dtypes)
    unknown_labels = set(participant_df["label"].dropna().unique()) - set(LABEL_MAPPING)
    if unknown_labels:
        raise ValueError(f"Rótulos desconhecidos em {path.name}: {sorted(unknown_labels)}")

    participant_df["label"] = participant_df["label"].map(LABEL_MAPPING).astype("string")
    participant_df.insert(1, "participant_id", np.int8(participant_id))
    frames.append(participant_df)
    source_report.append({
        "participant_id": participant_id,
        "participante": PARTICIPANT_NAMES[participant_id],
        "linhas": len(participant_df),
        "sessões": participant_df["session_id"].nunique(),
    })

df = pd.concat(frames, ignore_index=True)
del frames

overview = pd.Series({
    "linhas": len(df),
    "colunas": df.shape[1],
    "classes": df["label"].nunique(dropna=True),
    "participantes": df["participant_id"].nunique(dropna=True),
    "sessões": df[["participant_id", "session_id"]].drop_duplicates().shape[0],
    "memória_MB": df.memory_usage(deep=True).sum() / 1024**2,
}, name="valor")
display(pd.DataFrame(source_report).set_index("participant_id"))
display(overview.to_frame())
display(df[metadata_columns + coordinate_columns[:6]].head())
display(pd.crosstab(df["participant_id"], df["label"], margins=True))

## 2. Ausências, valores não finitos e duplicatas

Além de `NaN`, verificamos `+inf` e `-inf`. Duplicatas são avaliadas de duas formas:

- linha inteira repetida;
- chave lógica repetida (`participant_id`, `session_id`, `frame_index`).

Não removemos frames de sessões diferentes apenas por terem coordenadas iguais: em uma sequência de vídeo isso pode ser uma observação legítima.

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
columns_with_missing = missing_by_column[missing_by_column.gt(0)]

coordinate_values = df[coordinate_columns].to_numpy(dtype=np.float32, copy=False)
non_finite_row_mask = ~np.isfinite(coordinate_values).all(axis=1)
exact_duplicate_mask = df.duplicated(keep="first")
key_duplicate_mask = df.duplicated(
    subset=["participant_id", "session_id", "frame_index"], keep="first"
)

quality_report = pd.Series({
    "células_ausentes": int(missing_by_column.sum()),
    "linhas_com_ausências": int(df.isna().any(axis=1).sum()),
    "linhas_com_coordenadas_não_finitas": int(non_finite_row_mask.sum()),
    "duplicatas_exatas": int(exact_duplicate_mask.sum()),
    "chaves_session_frame_repetidas": int(key_duplicate_mask.sum()),
}, name="quantidade")

display(quality_report.to_frame())
if columns_with_missing.empty:
    print("Nenhuma coluna possui valores ausentes.")
else:
    display(columns_with_missing.rename("ausências").to_frame())

### Limpeza

Landmarks ausentes ou infinitos não são imputados, pois uma mediana criaria uma geometria facial artificial. Removemos essas linhas, registros sem metadados essenciais e ocorrências posteriores de uma mesma chave lógica.

In [ ]:
required_columns = metadata_columns + coordinate_columns
missing_required_row_mask = df[required_columns].isna().any(axis=1)
remove_mask = missing_required_row_mask | non_finite_row_mask | key_duplicate_mask

df_clean = (
    df.loc[~remove_mask]
    .sort_values(["participant_id", "session_id", "frame_index"], kind="stable")
    .reset_index(drop=True)
    .copy()
)

cleaning_report = pd.Series({
    "linhas_antes": len(df),
    "linhas_removidas": int(remove_mask.sum()),
    "linhas_depois": len(df_clean),
}, name="quantidade")
display(cleaning_report.to_frame())

assert not df_clean[required_columns].isna().any().any()
assert np.isfinite(df_clean[coordinate_columns].to_numpy(copy=False)).all()
assert not df_clean.duplicated(
    subset=["participant_id", "session_id", "frame_index"]
).any()

del coordinate_values

## 3. Normalização geométrica

Para cada frame:

1. subtraímos a ponta do nariz (landmark 1) de todos os pontos;
2. calculamos a distância 2D entre os cantos externos dos olhos (33 e 263);
3. dividimos `x`, `y` e `z` por essa escala;
4. rotacionamos `x` e `y` para deixar a linha entre os olhos horizontal.

O resultado mantém a geometria relativa do rosto. Não aplicamos `MinMaxScaler` ao conjunto completo porque isso aprenderia mínimos e máximos antes da separação entre treino e teste.

In [ ]:
LANDMARK_COUNT = 468
NOSE_TIP = 1
RIGHT_EYE_OUTER_CORNER = 33
LEFT_EYE_OUTER_CORNER = 263
EPSILON = 1e-6

points = np.ascontiguousarray(
    df_clean[coordinate_columns].to_numpy(dtype=np.float32, copy=True)
).reshape(-1, LANDMARK_COUNT, 3)

# Reancoragem por frame. No CSV atual esta operação deve ser praticamente nula.
points -= points[:, NOSE_TIP:NOSE_TIP + 1, :]
interocular_scale = np.linalg.norm(
    points[:, RIGHT_EYE_OUTER_CORNER, :2]
    - points[:, LEFT_EYE_OUTER_CORNER, :2],
    axis=1,
)
valid_scale_mask = np.isfinite(interocular_scale) & (interocular_scale > EPSILON)

if not valid_scale_mask.all():
    print(f"Removendo {(~valid_scale_mask).sum()} linha(s) com escala facial inválida.")
    df_clean = df_clean.loc[valid_scale_mask].reset_index(drop=True)
    points = points[valid_scale_mask]
    interocular_scale = interocular_scale[valid_scale_mask]

points /= interocular_scale[:, None, None]

# Corrige a rotação da cabeça no plano da imagem (roll).
eye_direction = (
    points[:, LEFT_EYE_OUTER_CORNER, :2]
    - points[:, RIGHT_EYE_OUTER_CORNER, :2]
)
face_roll_radians = np.arctan2(eye_direction[:, 1], eye_direction[:, 0])
cosine = np.cos(face_roll_radians)[:, None]
sine = np.sin(face_roll_radians)[:, None]
original_x = points[:, :, 0].copy()
original_y = points[:, :, 1].copy()
points[:, :, 0] = original_x * cosine + original_y * sine
points[:, :, 1] = -original_x * sine + original_y * cosine

normalized_coordinates = pd.DataFrame(
    points.reshape(len(points), -1),
    columns=coordinate_columns,
    dtype="float32",
)
df_normalized = pd.concat(
    [df_clean[metadata_columns].reset_index(drop=True), normalized_coordinates],
    axis=1,
)

normalization_check = pd.Series({
    "máximo_absoluto_na_ponta_do_nariz": float(np.abs(points[:, NOSE_TIP]).max()),
    "média_distância_interocular_normalizada": float(np.linalg.norm(
        points[:, RIGHT_EYE_OUTER_CORNER, :2]
        - points[:, LEFT_EYE_OUTER_CORNER, :2], axis=1
    ).mean()),
    "erro_vertical_médio_entre_cantos_dos_olhos": float(np.abs(
        points[:, RIGHT_EYE_OUTER_CORNER, 1]
        - points[:, LEFT_EYE_OUTER_CORNER, 1]
    ).mean()),
}, name="valor")
display(normalization_check.to_frame())

assert np.allclose(points[:, NOSE_TIP], 0.0, atol=1e-6)
assert np.allclose(
    np.linalg.norm(
        points[:, RIGHT_EYE_OUTER_CORNER, :2]
        - points[:, LEFT_EYE_OUTER_CORNER, :2], axis=1
    ),
    1.0,
    atol=1e-5,
)
assert np.allclose(
    points[:, RIGHT_EYE_OUTER_CORNER, 1],
    points[:, LEFT_EYE_OUTER_CORNER, 1],
    atol=1e-5,
)

## 4. Feature engineering

As distâncias abaixo são calculadas no rosto centralizado, escalado pela distância interocular e alinhado pela linha dos olhos. Por isso, distâncias e áreas são comparáveis entre frames e participantes. Os centros de sobrancelhas, olhos e bochechas são centróides de conjuntos de landmarks.

O CSV contém landmarks, não os pixels da imagem original. Por isso, o “pixel mais ao topo” é aproximado pelo landmark facial com menor valor de `y` em cada frame. Para a abertura total dos lábios, selecionamos dinamicamente o ponto mais alto do contorno superior e o ponto mais baixo do contorno inferior.

As features estão divididas em quatro grupos:

- **geométricas gerais:** relações entre sobrancelhas, bochechas, topo do rosto, lábios e queixo;
- **boca:** altura, largura, proporção e área da abertura;
- **sobrancelhas e olhos:** elevação vertical, simetria, inclinação, curvatura e abertura dos olhos;
- **temporais causais:** velocidade, aceleração e médias móveis calculadas somente dentro de cada sessão e usando o frame atual e anteriores.

Features relativas à mediana neutra do participante não são criadas aqui, pois dependeriam dos rótulos e de estatísticas calculadas antes da separação dos dados, criando risco de data leakage.

In [ ]:
# Índices do MediaPipe Face Mesh agrupados por região anatômica.
RIGHT_EYEBROW = np.array([46, 53, 52, 65, 55, 70, 63, 105, 66, 107])
LEFT_EYEBROW = np.array([276, 283, 282, 295, 285, 300, 293, 334, 296, 336])
RIGHT_EYE = np.array([33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246])
LEFT_EYE = np.array([263, 249, 390, 373, 374, 380, 381, 382, 362, 398, 384, 385, 386, 387, 388, 466])
RIGHT_CHEEK = np.array([50, 101, 205])
LEFT_CHEEK = np.array([280, 330, 425])
UPPER_LIP = np.array([0, 13, 37, 39, 40, 80, 81, 82, 185, 191, 267, 269, 270, 310, 311, 312, 409, 415])
LOWER_LIP = np.array([14, 17, 84, 87, 88, 91, 95, 146, 178, 181, 314, 317, 318, 321, 324, 375, 402, 405])
INNER_LIP_CONTOUR = np.array([78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308, 415, 310, 311, 312, 13, 82, 81, 80, 191])
MOUTH_VERTICAL_PAIRS = ((82, 87), (13, 14), (312, 317))
RIGHT_BROW_ENDPOINTS = (70, 107)
LEFT_BROW_ENDPOINTS = (336, 300)
RIGHT_BROW_ARCH_CENTER = 105
LEFT_BROW_ARCH_CENTER = 334
CHIN_TIP = 152

def centroid_2d(landmark_indices):
    return points[:, landmark_indices, :2].mean(axis=1)

def euclidean_distance_2d(first, second):
    return np.linalg.norm(first - second, axis=1)

def vertical_distance(first, second):
    return np.abs(first[:, 1] - second[:, 1])

def signed_angle_2d(first, second):
    difference = second - first
    return np.arctan2(difference[:, 1], difference[:, 0])

def polygon_area_2d(polygon):
    x_coordinates = polygon[:, :, 0]
    y_coordinates = polygon[:, :, 1]
    return 0.5 * np.abs(
        np.sum(
            x_coordinates * np.roll(y_coordinates, -1, axis=1)
            - y_coordinates * np.roll(x_coordinates, -1, axis=1),
            axis=1,
        )
    )

def eye_aspect_ratio(horizontal_pair, vertical_pairs):
    horizontal = euclidean_distance_2d(
        points[:, horizontal_pair[0], :2],
        points[:, horizontal_pair[1], :2],
    )
    vertical = sum(
        euclidean_distance_2d(points[:, upper, :2], points[:, lower, :2])
        for upper, lower in vertical_pairs
    )
    return vertical / (len(vertical_pairs) * horizontal + EPSILON)

def extreme_point_2d(landmark_indices, mode):
    region = points[:, landmark_indices, :2]
    if mode == "top":
        positions = region[:, :, 1].argmin(axis=1)
    elif mode == "bottom":
        positions = region[:, :, 1].argmax(axis=1)
    else:
        raise ValueError("mode deve ser 'top' ou 'bottom'")
    return region[np.arange(len(region)), positions]

right_eyebrow_center = centroid_2d(RIGHT_EYEBROW)
left_eyebrow_center = centroid_2d(LEFT_EYEBROW)
eyebrows_center = (right_eyebrow_center + left_eyebrow_center) / 2

right_eye_center = centroid_2d(RIGHT_EYE)
left_eye_center = centroid_2d(LEFT_EYE)
right_cheek_center = centroid_2d(RIGHT_CHEEK)
left_cheek_center = centroid_2d(LEFT_CHEEK)
cheeks_center = (right_cheek_center + left_cheek_center) / 2

top_face_positions = points[:, :, 1].argmin(axis=1)
top_face_point = points[np.arange(len(points)), top_face_positions, :2]
top_upper_lip_point = extreme_point_2d(UPPER_LIP, mode="top")
bottom_lower_lip_point = extreme_point_2d(LOWER_LIP, mode="bottom")
chin_tip_point = points[:, CHIN_TIP, :2]

In [ ]:
right_eyebrow_eye_distance = euclidean_distance_2d(
    right_eyebrow_center, right_eye_center
)
left_eyebrow_eye_distance = euclidean_distance_2d(
    left_eyebrow_center, left_eye_center
)

# Medidas específicas da boca.
mouth_vertical_height = vertical_distance(
    points[:, 13, :2], points[:, 14, :2]
)
mouth_width = euclidean_distance_2d(
    points[:, 61, :2], points[:, 291, :2]
)
mouth_vertical_gaps = np.column_stack([
    vertical_distance(points[:, upper, :2], points[:, lower, :2])
    for upper, lower in MOUTH_VERTICAL_PAIRS
])
mouth_average_height = mouth_vertical_gaps.mean(axis=1)
mouth_aspect_ratio = mouth_average_height / (mouth_width + EPSILON)
mouth_open_area = polygon_area_2d(points[:, INNER_LIP_CONTOUR, :2])

# Elevação vertical das sobrancelhas em relação aos olhos.
right_brow_vertical = vertical_distance(right_eye_center, right_eyebrow_center)
left_brow_vertical = vertical_distance(left_eye_center, left_eyebrow_center)
mean_brow_vertical = (right_brow_vertical + left_brow_vertical) / 2
brow_asymmetry = np.abs(right_brow_vertical - left_brow_vertical)

right_brow_angle = signed_angle_2d(
    points[:, RIGHT_BROW_ENDPOINTS[0], :2],
    points[:, RIGHT_BROW_ENDPOINTS[1], :2],
)
left_brow_angle = signed_angle_2d(
    points[:, LEFT_BROW_ENDPOINTS[0], :2],
    points[:, LEFT_BROW_ENDPOINTS[1], :2],
)
right_brow_curvature = (
    points[:, list(RIGHT_BROW_ENDPOINTS), 1].mean(axis=1)
    - points[:, RIGHT_BROW_ARCH_CENTER, 1]
)
left_brow_curvature = (
    points[:, list(LEFT_BROW_ENDPOINTS), 1].mean(axis=1)
    - points[:, LEFT_BROW_ARCH_CENTER, 1]
)

right_eye_aspect_ratio = eye_aspect_ratio(
    (33, 133), ((159, 145), (158, 153))
)
left_eye_aspect_ratio = eye_aspect_ratio(
    (362, 263), ((386, 374), (385, 380))
)
mean_eye_aspect_ratio = (right_eye_aspect_ratio + left_eye_aspect_ratio) / 2

engineered_features = pd.DataFrame({
    "dist_centro_sobrancelhas_centro_bochechas": euclidean_distance_2d(
        eyebrows_center, cheeks_center
    ),
    "dist_centro_sobrancelhas_topo_rosto": euclidean_distance_2d(
        eyebrows_center, top_face_point
    ),
    "dist_sobrancelha_direita_olho_direito": right_eyebrow_eye_distance,
    "dist_sobrancelha_esquerda_olho_esquerdo": left_eyebrow_eye_distance,
    "dist_media_sobrancelhas_olhos": (
        right_eyebrow_eye_distance + left_eyebrow_eye_distance
    ) / 2,
    "dist_topo_labio_superior_base_labio_inferior": euclidean_distance_2d(
        top_upper_lip_point, bottom_lower_lip_point
    ),
    "dist_topo_labio_superior_ponta_queixo": euclidean_distance_2d(
        top_upper_lip_point, chin_tip_point
    ),
    "altura_vertical_boca": mouth_vertical_height,
    "largura_boca": mouth_width,
    "altura_media_boca": mouth_average_height,
    "mouth_aspect_ratio": mouth_aspect_ratio,
    "area_abertura_boca": mouth_open_area,
    "dist_vertical_sobrancelha_direita_olho": right_brow_vertical,
    "dist_vertical_sobrancelha_esquerda_olho": left_brow_vertical,
    "dist_vertical_media_sobrancelhas_olhos": mean_brow_vertical,
    "assimetria_sobrancelhas": brow_asymmetry,
    "inclinacao_sobrancelha_direita": right_brow_angle,
    "inclinacao_sobrancelha_esquerda": left_brow_angle,
    "curvatura_sobrancelha_direita": right_brow_curvature,
    "curvatura_sobrancelha_esquerda": left_brow_curvature,
    "eye_aspect_ratio_direito": right_eye_aspect_ratio,
    "eye_aspect_ratio_esquerdo": left_eye_aspect_ratio,
    "eye_aspect_ratio_medio": mean_eye_aspect_ratio,
}, dtype="float32")

# Features temporais causais: somente o presente e frames anteriores da mesma sessão.
temporal_context = df_normalized[["participant_id", "session_id", "frame_index"]].copy()
temporal_context["abertura_boca"] = mouth_average_height
temporal_context["sobrancelha_direita"] = right_brow_vertical
temporal_context["sobrancelha_esquerda"] = left_brow_vertical
temporal_group_columns = ["participant_id", "session_id"]
temporal_groups = temporal_context.groupby(temporal_group_columns, sort=False)
frame_step = temporal_groups["frame_index"].diff().astype("float32")
frame_step = frame_step.where(frame_step > 0)

def causal_velocity(column):
    difference = temporal_groups[column].diff()
    return (difference / frame_step).fillna(0.0).astype("float32")

def causal_trailing_mean(column, window=5):
    return (
        temporal_context.groupby(temporal_group_columns, sort=False)[column]
        .transform(lambda values: values.rolling(window, min_periods=1).mean())
        .astype("float32")
    )

temporal_context["velocidade_abertura_boca"] = causal_velocity("abertura_boca")
temporal_context["velocidade_sobrancelha_direita"] = causal_velocity("sobrancelha_direita")
temporal_context["velocidade_sobrancelha_esquerda"] = causal_velocity("sobrancelha_esquerda")
temporal_context["aceleracao_abertura_boca"] = (
    temporal_context.groupby(temporal_group_columns, sort=False)["velocidade_abertura_boca"]
    .diff()
    .div(frame_step)
    .fillna(0.0)
    .astype("float32")
)

engineered_features["velocidade_abertura_boca"] = temporal_context["velocidade_abertura_boca"]
engineered_features["aceleracao_abertura_boca"] = temporal_context["aceleracao_abertura_boca"]
engineered_features["velocidade_sobrancelha_direita"] = temporal_context["velocidade_sobrancelha_direita"]
engineered_features["velocidade_sobrancelha_esquerda"] = temporal_context["velocidade_sobrancelha_esquerda"]
engineered_features["media_movel_5_abertura_boca"] = causal_trailing_mean("abertura_boca")
engineered_features["media_movel_5_sobrancelha_direita"] = causal_trailing_mean("sobrancelha_direita")
engineered_features["media_movel_5_sobrancelha_esquerda"] = causal_trailing_mean("sobrancelha_esquerda")

prepared_data = pd.concat([df_normalized, engineered_features], axis=1)
feature_columns = engineered_features.columns.tolist()

FEATURE_DOCUMENTATION = [
    ("geral", "dist_centro_sobrancelhas_centro_bochechas", "Distância 2D entre o centro conjunto das sobrancelhas e o centro conjunto das bochechas."),
    ("geral", "dist_centro_sobrancelhas_topo_rosto", "Distância 2D entre o centro das sobrancelhas e o landmark mais alto do rosto."),
    ("geral", "dist_sobrancelha_direita_olho_direito", "Distância 2D entre os centroides da sobrancelha e do olho direitos."),
    ("geral", "dist_sobrancelha_esquerda_olho_esquerdo", "Distância 2D entre os centroides da sobrancelha e do olho esquerdos."),
    ("geral", "dist_media_sobrancelhas_olhos", "Média das distâncias 2D entre cada sobrancelha e seu respectivo olho."),
    ("geral", "dist_topo_labio_superior_base_labio_inferior", "Distância 2D entre o ponto mais alto do lábio superior e o mais baixo do lábio inferior."),
    ("geral", "dist_topo_labio_superior_ponta_queixo", "Distância 2D entre o ponto mais alto do lábio superior e a ponta do queixo."),
    ("boca", "altura_vertical_boca", "Abertura vertical central entre os landmarks internos 13 e 14."),
    ("boca", "largura_boca", "Distância 2D entre os cantos da boca, landmarks 61 e 291."),
    ("boca", "altura_media_boca", "Média de três aberturas verticais internas: 82–87, 13–14 e 312–317."),
    ("boca", "mouth_aspect_ratio", "Razão entre a altura média interna e a largura da boca."),
    ("boca", "area_abertura_boca", "Área do polígono formado pelo contorno interno dos lábios."),
    ("sobrancelhas", "dist_vertical_sobrancelha_direita_olho", "Separação somente no eixo vertical entre os centroides da sobrancelha e do olho direitos."),
    ("sobrancelhas", "dist_vertical_sobrancelha_esquerda_olho", "Separação somente no eixo vertical entre os centroides da sobrancelha e do olho esquerdos."),
    ("sobrancelhas", "dist_vertical_media_sobrancelhas_olhos", "Média das separações verticais das duas sobrancelhas em relação aos olhos."),
    ("sobrancelhas", "assimetria_sobrancelhas", "Diferença absoluta entre as elevações verticais direita e esquerda."),
    ("sobrancelhas", "inclinacao_sobrancelha_direita", "Ângulo em radianos entre as extremidades da sobrancelha direita após alinhamento dos olhos."),
    ("sobrancelhas", "inclinacao_sobrancelha_esquerda", "Ângulo em radianos entre as extremidades da sobrancelha esquerda após alinhamento dos olhos."),
    ("sobrancelhas", "curvatura_sobrancelha_direita", "Altura do centro do arco direito em relação à média vertical de suas extremidades."),
    ("sobrancelhas", "curvatura_sobrancelha_esquerda", "Altura do centro do arco esquerdo em relação à média vertical de suas extremidades."),
    ("olhos", "eye_aspect_ratio_direito", "Razão entre a abertura vertical e a largura do olho direito."),
    ("olhos", "eye_aspect_ratio_esquerdo", "Razão entre a abertura vertical e a largura do olho esquerdo."),
    ("olhos", "eye_aspect_ratio_medio", "Média dos aspect ratios dos dois olhos."),
    ("temporal", "velocidade_abertura_boca", "Variação causal da altura média da boca por frame, reiniciada em cada sessão."),
    ("temporal", "aceleracao_abertura_boca", "Variação causal da velocidade de abertura da boca por frame."),
    ("temporal", "velocidade_sobrancelha_direita", "Variação causal da elevação vertical da sobrancelha direita por frame."),
    ("temporal", "velocidade_sobrancelha_esquerda", "Variação causal da elevação vertical da sobrancelha esquerda por frame."),
    ("temporal", "media_movel_5_abertura_boca", "Média da abertura da boca no frame atual e em até quatro frames anteriores da mesma sessão."),
    ("temporal", "media_movel_5_sobrancelha_direita", "Média da elevação direita no frame atual e em até quatro frames anteriores da mesma sessão."),
    ("temporal", "media_movel_5_sobrancelha_esquerda", "Média da elevação esquerda no frame atual e em até quatro frames anteriores da mesma sessão."),
]
feature_documentation = pd.DataFrame(
    FEATURE_DOCUMENTATION, columns=["grupo", "feature", "explicação"]
)
assert set(feature_documentation["feature"]) == set(feature_columns)

display(feature_documentation)
display(prepared_data[metadata_columns + feature_columns].head())
display(prepared_data[feature_columns].describe().T)

## 5. Validação e conjunto pronto

A célula seguinte confirma a integridade do resultado e cria `X`, `y` e dois agrupamentos. Para avaliar generalização a pessoas não vistas, use `participant_groups`; para avaliar apenas novas gravações das mesmas pessoas, use `session_groups`. `participant_id`, `session_id`, `frame_index` e `label` nunca entram em `X`. As features temporais são causais e reiniciadas em cada sessão, mas a divisão do modelo ainda deve respeitar pelo menos `session_groups`. Não foram adicionadas features construídas com rótulos ou baselines neutros. Se o modelo exigir padronização estatística, ajuste `StandardScaler` **somente em `X_train`**, dentro de um `Pipeline`.

In [ ]:
assert len(prepared_data) == len(df_normalized) == len(engineered_features)
assert not prepared_data.isna().any().any()
assert np.isfinite(prepared_data[coordinate_columns + feature_columns].to_numpy()).all()
session_start_mask = ~prepared_data.duplicated(
    subset=["participant_id", "session_id"], keep="first"
)
causal_derivative_columns = [
    "velocidade_abertura_boca",
    "aceleracao_abertura_boca",
    "velocidade_sobrancelha_direita",
    "velocidade_sobrancelha_esquerda",
]
assert np.allclose(
    prepared_data.loc[session_start_mask, causal_derivative_columns], 0.0
)

X = prepared_data.drop(columns=metadata_columns)
y = prepared_data["label"].copy()
participant_groups = prepared_data["participant_id"].copy()
session_groups = (
    prepared_data["participant_id"].astype(str)
    + "::"
    + prepared_data["session_id"].astype(str)
)

final_report = pd.Series({
    "amostras": len(prepared_data),
    "atributos_em_X": X.shape[1],
    "features_criadas": len(feature_columns),
    "classes": y.nunique(),
    "participantes": participant_groups.nunique(),
    "sessões_para_agrupamento": session_groups.nunique(),
}, name="valor")
display(final_report.to_frame())
display(pd.crosstab(prepared_data["participant_id"], y, margins=True))
display(
    prepared_data.groupby("label", observed=True)[feature_columns]
    .mean()
    .round(4)
)

## 6. Exportação

O arquivo unificado inclui `participant_id` e é salvo em `data/prepared_landmarks.csv`. A opção abaixo permanece explícita para evitar sobrescritas acidentais em futuras execuções.

In [ ]:
SAVE_PREPARED_DATA = True  # Defina como True para salvar o CSV preparado, ou False para não salvar.

if SAVE_PREPARED_DATA:
    prepared_data.to_csv(OUTPUT_PATH, index=False)
    print(f"Dados preparados salvos em: {OUTPUT_PATH}")
else:
    print("Exportação desativada. Defina SAVE_PREPARED_DATA = True para salvar o CSV.")